# 🚀 [Phase 3] 구글 코랩(T4) 무거운 트랜스포머 풀학습
이 노트북은 파라미터가 거대한 모델들(WavLM, Wav2Vec2, HuBERT)을 코랩에서 100% 데이터로 훈련하기 위한 전용 노트북입니다.

In [ ]:
import os, glob, subprocess, sys
from google.colab import drive

# 1. 구글 드라이브 마운트 (데이터 및 가중치 저장용)
try:
    drive.mount('/content/drive')
except:
    pass

# 2. 코랩 로컬(SSD)에 최신 코드 자동 다운로드 (GitHub)
print("📥 깃허브에서 최신 코드를 코랩으로 다운로드합니다...")
if not os.path.exists('/content/DCC_Problem'):
    subprocess.run(["git", "clone", "-b", "mission2", "https://github.com/KozzilzzilE/DCC_Problem.git", "/content/DCC_Problem"])
else:
    subprocess.run(["git", "-C", "/content/DCC_Problem", "pull", "origin", "mission2"])

sys.path.append('/content/DCC_Problem/mission2_speaker')
os.chdir('/content/DCC_Problem/mission2_speaker')

# 3. 필수 라이브러리 설치
print("📦 라이브러리 설치 중...")
!pip install -q librosa soundfile transformers torchaudio scikit-learn tabulate pandas matplotlib

# 4. 드라이브에서 데이터 로컬(SSD)로 압축 해제
DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)
DRIVE_ROOT = "/content/drive/MyDrive/DCC"
dcc_zips = glob.glob(f"{DRIVE_ROOT}/data/**/*.zip", recursive=True)
if dcc_zips:
    print(f"📂 구글 드라이브에서 압축 파일 {len(dcc_zips)}개 발견! 코랩 초고속 SSD로 압축 해제합니다 (약 1~2분 소요)...")
    for zf in dcc_zips:
        subprocess.run(["7z", "x", zf, f"-o{DATA_ROOT}", "-y"], stdout=subprocess.DEVNULL)
    print("✅ 압축 해제 완료!")
else:
    print("⚠️ 경고: 구글 드라이브(MyDrive/DCC/data)에서 zip 파일을 찾을 수 없습니다! 드라이브 마운트를 확인하세요.")

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
OUTPUT_DIR = f"{DRIVE_ROOT}/mission2_speaker/test_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ 모든 환경 세팅이 완벽하게 끝났습니다!")


In [ ]:
import torch
from benchmark_suite.dataset import UniversalSpeakerDataset
from benchmark_suite.models import build_model
from benchmark_suite.trainer import BenchmarkTrainer
from benchmark_suite.config import MODEL_REGISTRY
from torch.utils.data import DataLoader

trainer = BenchmarkTrainer(
    output_dir=OUTPUT_DIR,
    drive_backup_dir=OUTPUT_DIR
)

PHASE2_MODELS = ["wavlm", "wav2vec2", "hubert"]

for model_name in PHASE2_MODELS:
    cfg = MODEL_REGISTRY.get(model_name, {})
    
    train_dataset = UniversalSpeakerDataset(TRAIN_DIR, model_name=model_name, config_dict=cfg, is_train=True)
    val_dataset = UniversalSpeakerDataset(VAL_DIR, model_name=model_name, config_dict=cfg, is_train=False)
    
    # 트랜스포머는 VRAM을 많이 먹으므로 batch_size=8
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
    
    model = build_model(model_name, num_classes=2, pretrained=True)
    
    trainer.fit(
        model_name=model_name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=15,
        lr=1e-4,
        skip_if_done=False
    )

print("✅ 코랩 무거운 모델 학습 완료!")
